In [ ]:
''' Bibliotecas utilizadas '''
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

''' Funções desenvolvidos para executar o experimento '''
from executa_experimento import ExecutaExperimento
from settings import set_configs

In [ ]:
''' eICU-CRD: Banco de dados de UTIs com registros clínicos anonimizados de mais de 200 hospitais dos EUA. Mantido pela PhysioNet.''' 

''' Arquivos do eICU-CRD usados para prever o delirium '''
diagnosis = pd.read_csv("diagnosis.csv")
patient = pd.read_csv("patient.csv")
apache = pd.read_csv("apachePatientResult.csv")
lab = pd.read_csv("lab.csv")

In [ ]:
''' Monta o dataset com as informações das três planilhas anteriores '''

# Termos que foram considerados delirium (classe positiva)
delirium_terms = ['delirium', 'encephalopathy', 'acute confusional']
# Cria a coluna com as classes positiva (ocorrência de delirium) e negativa (não houve occorrência de delirium)
diagnosis['delirium'] = diagnosis['diagnosisstring'].str.lower().str.contains('|'.join(delirium_terms))
# Cria um dataframe com 1 linha por paciente, indicando se teve delirium
delirium_labels = diagnosis.groupby('patientunitstayid')['delirium'].max().reset_index()

# Verifica se o paciente tem hipertensão 
diagnosis['hypertension'] = diagnosis['diagnosisstring'].str.lower().str.contains('hypertension')
hypertension_labels = diagnosis.groupby('patientunitstayid')['hypertension'].max().reset_index()

# Laboratoriais selecionados (usando a primeira medição)
labs_to_extract = {
    'chloride': 'chloride',
    'creatinine': 'creatinine',
    'sodium': 'sodium',
    'albumin': 'albumin',
    'bun': 'urea',
    'lactate': 'lactate'
}
lab_filtered = lab[lab['labname'].str.lower().isin(labs_to_extract.keys())]
lab_filtered['labname'] = lab_filtered['labname'].str.lower()

# Pegar a primeira medição por paciente por exame
lab_pivot = (
    lab_filtered.sort_values(['patientunitstayid', 'labname', 'labresultoffset'])
    .groupby(['patientunitstayid', 'labname'])['labresult']
    .first()
    .unstack()
    .rename(columns=labs_to_extract)
    .reset_index()
)

# Faz o merge das planilhas
df = patient.merge(delirium_labels, on='patientunitstayid', how='left')
df = df.merge(hypertension_labels, on='patientunitstayid', how='left')
df['delirium'] = df['delirium'].fillna(0).astype(int)
df['hypertension'] = df['hypertension'].fillna(0).astype(int)
# Merge com colunas úteis do APACHE
# APACHE = Acute Physiology and Chronic Health Evaluation
# É um sistema de escore usado para estimar a gravidade da condição de pacientes em UTI. 
# Ele considera: Parâmetros fisiológicos (ex: pressão arterial, temperatura, creatinina, etc.), Idade, Condições clínicas pré-existentes.
df = df.merge(apache[['patientunitstayid', 'acutephysiologyscore', 'apachescore',
                      'predictedicumortality', 'predictediculos']], on='patientunitstayid', how='left')
df = df.merge(lab_pivot, on='patientunitstayid', how='left')

In [ ]:
''' Seleciona as colunas relevantes para a predição de delirium '''

selected_columns = [
    'age', 'gender', 'ethnicity', 'unittype', 'unitadmitsource', 'unitstaytype', 'admissionweight', 
    'acutephysiologyscore', 'apachescore', 'predictedicumortality', 'predictediculos', 'chloride', 
    'creatinine', 'sodium', 'albumin', 'urea', 'lactate', 'hypertension', 'delirium'
]

'''
age : Idade do paciente em anos.
gender : Gênero do paciente, geralmente "M" (masculino) ou "F" (feminino).
ethnicity : Etnia autorrelatada ou registrada no prontuário.
unittype : Tipo da UTI em que o paciente está internado (ex: Med-Surg ICU, Neuro ICU, Cardiac ICU).
unitadmitsource : Origem do paciente ao ser admitido na UTI (ex: emergência, bloco cirúrgico, outra unidade hospitalar).
unitstaytype : Tipo da permanência na UTI, como admissão direta, transferência interna, etc.
admissionweight : Peso do paciente no momento da admissão (em kg).
acutephysiologyscore : Escore de fisiologia aguda do sistema APACHE. 
                       Soma de pontos baseados em alterações em sinais vitais e exames laboratoriais nas primeiras horas da admissão.
apachescore : score APACHE global, que incorpora o acutephysiologyscore, idade e presença de doenças crônicas.
predictedicumortality : Probabilidade (entre 0 e 1) de morte na UTI, estimada com base nos parâmetros do APACHE. 
                        (Reflete a estimativa geral de gravidade.)
predictediculos : Tempo estimado de permanência na UTI (em dias), baseado no escore clínico inicial.
chloride : nível de cloro no sangue  (mEq/L). (Distúrbios eletrolíticos estão associados a alterações do estado mental.)
creatinine : nível de creatinina no sangue (mg/dL). (Reflete a função renal. Insuficiência renal está associada a maior risco de delirium.)
sodium : concentração de sódio no sangue (mEq/L). (Hiponatremia ou hipernatremia são causas comuns de confusão mental e delirium.)
albumin : nível de albumina no sangue (g/dL). (Baixos níveis aumentam o risco de complicações.)
urea : Nível de ureia no sangue (mg/dL). (Acúmulo de ureia está associado à encefalopatia urêmica, que pode se manifestar como delirium.)
lactate : Nível de lactato no sangue (mmol/L). (Lactato elevado pode indicar risco iminente de falência orgânica e delirium.)
hypertension : Variável binária que indica se o paciente tem diagnóstico de hipertensão (0 = não, 1 = sim). 
(Hipertensão está associada a risco vascular aumentado, que pode contribuir para vulnerabilidade ao delirium.)
delirium : classe a ser predita conforme transformações descritas acima.
'''

# Transforma a coluna 'age' 
df["age"] = df["age"].replace("> 89", "90")  # substitui '>89' por '90'
df["age"] = pd.to_numeric(df["age"], errors="coerce")  # converte para número
# Seleciona as colunas 
df_ml = df[selected_columns].dropna()
df_ml = df_ml.reset_index(drop=True)

In [ ]:
''' Exibe o dataset (antes do pré-processamento) '''
df_ml

In [ ]:
''' Executa o pré-processamento  '''

# Separa x e y
X = df_ml.drop("delirium", axis=1)
y = df_ml["delirium"]

# Identifica tipos de variáveis
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

# Cria transformadores
numeric_transformer = Pipeline(steps=[
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

# Cria pré-processador
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

# Aplica pré-processamento
X_processed = preprocessor.fit_transform(X)

# Obtem nomes das colunas pós-transformação
cat_columns = preprocessor.named_transformers_["cat"]["onehot"].get_feature_names_out(categorical_features)
all_columns = numeric_features + cat_columns.tolist()

# Cria a matriz de atributos
X_final = pd.DataFrame(X_processed, columns=all_columns) 

In [ ]:
''' Exibe o dataset (depois do pré-processamento) '''
X_final

In [ ]:
''' Executa o experimento '''
# ExecutaExperimento : o código está em executa_experimento.py
# set_configs : algoritmos e configurações a serem testadas (arquivo: settings.py)
# n = 5 : aplica k-fold cross-validation com k = 5
# nome_experimento : nome de como vai salvar os resultados encontrados neste experimento. 
experimento = ExecutaExperimento(set_configs, n=5, nome_experimento='experimento-final')
experimento.executa(X_final, y)